# Experiment Tracking

This notebook demonstrates a structured approach to experiment tracking. We run multiple model configurations, log their hyperparameters and results, and compare them side by side in a summary table.

**Experiments:**
- **v1** — Baseline MLP (vanilla architecture, SGD optimizer)
- **v2** — Improved model (BatchNorm, Dropout, AdamW optimizer, learning rate scheduling)

## 1. Experiment Tracking Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import time
import copy
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Experiment registry
EXPERIMENTS = []

def log_experiment(exp_id, config, results):
    """Register an experiment with its config and results."""
    entry = {'experiment': exp_id}
    entry.update(config)
    entry.update(results)
    EXPERIMENTS.append(entry)
    print(f'Experiment {exp_id} logged: test_acc={results["test_accuracy"]:.4f}, f1={results["test_f1"]:.4f}')

In [ ]:
# Shared data loading
train_df = pd.read_csv('../datasets/train.csv')
val_df = pd.read_csv('../datasets/validation.csv')
test_df = pd.read_csv('../datasets/test.csv')
target_col = 'target'

le = LabelEncoder()
y_train = le.fit_transform(train_df[target_col].values)
y_val = le.transform(val_df[target_col].values)
y_test = le.transform(test_df[target_col].values)
num_classes = len(le.classes_)

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df.drop(columns=[target_col]).values.astype(np.float32))
X_val = scaler.transform(val_df.drop(columns=[target_col]).values.astype(np.float32))
X_test = scaler.transform(test_df.drop(columns=[target_col]).values.astype(np.float32))
input_dim = X_train.shape[1]

train_loader = DataLoader(TensorDataset(
    torch.from_numpy(X_train), torch.from_numpy(y_train).long()
), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(
    torch.from_numpy(X_val), torch.from_numpy(y_val).long()
), batch_size=256, shuffle=False)
test_loader = DataLoader(TensorDataset(
    torch.from_numpy(X_test), torch.from_numpy(y_test).long()
), batch_size=256, shuffle=False)

print(f'Data loaded: {input_dim} features, {num_classes} classes')

## 2. Experiment v1 — Baseline MLP

In [ ]:
class BaselineMLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=128):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.network(x)

# Configuration
v1_config = {
    'model': 'BaselineMLP',
    'hidden_dim': 128,
    'optimizer': 'SGD',
    'lr': 0.01,
    'batch_norm': False,
    'dropout': 0.0,
    'lr_scheduler': False
}

# Train
model_v1 = BaselineMLP(input_dim, num_classes).to(device)
optimizer_v1 = optim.SGD(model_v1.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

print(f'Model v1 parameters: {sum(p.numel() for p in model_v1.parameters()):,}')

best_val_loss = float('inf')
best_state = None
start = time.time()

for epoch in range(1, 51):
    model_v1.train()
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer_v1.zero_grad()
        loss = criterion(model_v1(X_b), y_b)
        loss.backward()
        optimizer_v1.step()

    model_v1.eval()
    vloss = 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            vloss += criterion(model_v1(X_b), y_b).item() * X_b.size(0)
    vloss /= len(val_loader.dataset)

    if vloss < best_val_loss:
        best_val_loss = vloss
        best_state = copy.deepcopy(model_v1.state_dict())

elapsed = time.time() - start
model_v1.load_state_dict(best_state)

# Evaluate
model_v1.eval()
preds, labels = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        _, p = torch.max(model_v1(X_b), 1)
        preds.extend(p.cpu().numpy())
        labels.extend(y_b.numpy())

v1_results = {
    'test_accuracy': accuracy_score(labels, preds),
    'test_f1': f1_score(labels, preds, average='weighted'),
    'test_precision': precision_score(labels, preds, average='weighted'),
    'test_recall': recall_score(labels, preds, average='weighted'),
    'train_time_sec': round(elapsed, 1),
    'best_val_loss': round(best_val_loss, 4)
}

log_experiment('v1_baseline', v1_config, v1_results)

## 3. Experiment v2 — Improved (BatchNorm + Dropout + AdamW)

In [ ]:
class ImprovedMLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dims=[256, 128, 64], dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Configuration
v2_config = {
    'model': 'ImprovedMLP',
    'hidden_dims': [256, 128, 64],
    'optimizer': 'AdamW',
    'lr': 0.001,
    'batch_norm': True,
    'dropout': 0.3,
    'lr_scheduler': True,
    'weight_decay': 1e-4
}

# Train
model_v2 = ImprovedMLP(input_dim, num_classes).to(device)
optimizer_v2 = optim.AdamW(model_v2.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_v2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_v2, mode='min', factor=0.5, patience=5)

print(f'Model v2 parameters: {sum(p.numel() for p in model_v2.parameters()):,}')

best_val_loss = float('inf')
best_state = None
patience_counter = 0
start = time.time()

for epoch in range(1, 101):
    model_v2.train()
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer_v2.zero_grad()
        loss = criterion(model_v2(X_b), y_b)
        loss.backward()
        optimizer_v2.step()

    model_v2.eval()
    vloss = 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            vloss += criterion(model_v2(X_b), y_b).item() * X_b.size(0)
    vloss /= len(val_loader.dataset)
    scheduler_v2.step(vloss)

    if vloss < best_val_loss:
        best_val_loss = vloss
        best_state = copy.deepcopy(model_v2.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
    if patience_counter >= 10:
        break

elapsed = time.time() - start
model_v2.load_state_dict(best_state)

# Evaluate
model_v2.eval()
preds, labels = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        _, p = torch.max(model_v2(X_b), 1)
        preds.extend(p.cpu().numpy())
        labels.extend(y_b.numpy())

v2_results = {
    'test_accuracy': accuracy_score(labels, preds),
    'test_f1': f1_score(labels, preds, average='weighted'),
    'test_precision': precision_score(labels, preds, average='weighted'),
    'test_recall': recall_score(labels, preds, average='weighted'),
    'train_time_sec': round(elapsed, 1),
    'best_val_loss': round(best_val_loss, 4)
}

log_experiment('v2_improved', v2_config, v2_results)

## 4. Experiment Comparison Table

In [ ]:
comparison_df = pd.DataFrame(EXPERIMENTS)
comparison_df = comparison_df.set_index('experiment')

# Format columns for readability
display_cols = [
    'model', 'optimizer', 'lr', 'batch_norm', 'dropout', 'lr_scheduler',
    'test_accuracy', 'test_f1', 'test_precision', 'test_recall',
    'train_time_sec', 'best_val_loss'
]
existing_cols = [c for c in display_cols if c in comparison_df.columns]

print('=' * 90)
print('EXPERIMENT COMPARISON')
print('=' * 90)
comparison_df[existing_cols].T

In [ ]:
# Save comparison results
comparison_df[existing_cols].to_csv('../results/experiment_comparison.csv')
with open('../results/experiments.json', 'w') as f:
    json.dump(EXPERIMENTS, f, indent=2)

# Highlight best experiment
best_idx = comparison_df['test_accuracy'].idxmax()
best_acc = comparison_df.loc[best_idx, 'test_accuracy']
best_f1 = comparison_df.loc[best_idx, 'test_f1']
print(f'\nBest experiment: {best_idx}')
print(f'  Accuracy: {best_acc:.4f}')
print(f'  F1 Score: {best_f1:.4f}')
print('\nExperiment comparison saved.')